# Online Voice-Enabled Chatbot Using Speech Recognition and Deep Learning
### Academic Project: Deep Learning Intent Classification Pipeline
This notebook demonstrates the end-to-end deep learning pipeline for conversational intent classification:
1. **Data Loading & Exploratory Data Analysis (EDA)**
2. **Text Normalization & Tokenization**
3. **Sequence Vectorization & Padding**
4. **Deep Neural Network Construction (Bi-LSTM)**
5. **Model Training & Hyperparameter Tuning**
6. **Evaluation: Accuracy, Precision, Recall, F1-Score & Confusion Matrix**
7. **Inference with Confidence Thresholding & Fallback Handling**

In [ ]:
import os
import json
import re
import string
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print('TensorFlow Version:', tf.__version__)

## 1. Load Dataset & Exploratory Data Analysis

In [ ]:
data_path = os.path.join('..', 'data', 'intents.json')
with open(data_path, 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

patterns = []
tags = []
responses = {}

for intent in intents_data['intents']:
    tag = intent['tag']
    responses[tag] = intent['responses']
    for p in intent['patterns']:
        patterns.append(p)
        tags.append(tag)

print(f'Total Patterns: {len(patterns)}')
print(f'Total Unique Intents: {len(set(tags))}')

## 2. Text Preprocessing & Tokenization

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"won't", 'will not', text)
    text = re.sub(r"can't", 'can not', text)
    text = re.sub(r"n't", ' not', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_patterns = [clean_text(p) for p in patterns]

MAX_VOCAB = 2000
MAX_LEN = 16

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(cleaned_patterns)
sequences = tokenizer.texts_to_sequences(cleaned_patterns)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(tags)
y = tf.keras.utils.to_categorical(y_int)

print('X shape:', X.shape)
print('y shape:', y.shape)

## 3. Train / Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y_int
)
print(f'Train samples: {len(X_train)} | Val samples: {len(X_val)}')

## 4. Deep Learning Model Architecture (Bi-LSTM)

In [ ]:
vocab_size = len(tokenizer.word_index) + 2
num_classes = y.shape[1]

model = Sequential([
    Input(shape=(MAX_LEN,), name='input_sequence'),
    Embedding(input_dim=vocab_size, output_dim=64, name='embedding_layer'),
    Bidirectional(LSTM(48, return_sequences=True), name='bilstm_layer'),
    GlobalMaxPooling1D(name='pooling_layer'),
    Dense(64, activation='relu', name='dense_layer'),
    Dropout(0.3, name='dropout_regularization'),
    Dense(num_classes, activation='softmax', name='output_layer')
], name='VoiceChatbot_BiLSTM')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 5. Model Training

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=8,
    verbose=1
)

## 6. Evaluation & Confusion Matrix

In [ ]:
y_val_true = np.argmax(y_val, axis=1)
y_val_pred = np.argmax(model.predict(X_val), axis=1)

print(classification_report(y_val_true, y_val_pred, target_names=label_encoder.classes_))

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_val_true, y_val_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Interactive Prediction Pipeline

In [ ]:
import random
def predict_query(text, threshold=0.35):
    cleaned = clean_text(text)
    seq = pad_sequences(tokenizer.texts_to_sequences([cleaned]), maxlen=MAX_LEN, padding='post')
    probs = model.predict(seq, verbose=0)[0]
    top_idx = int(np.argmax(probs))
    conf = probs[top_idx]
    tag = label_encoder.classes_[top_idx]
    if conf < threshold:
        tag = 'fallback'
    resp = random.choice(responses.get(tag, ['I am not sure.']))
    return tag, conf, resp

for sample in ['What are your working hours?', 'Where is your office?', 'Tell me a joke']:
    tag, conf, resp = predict_query(sample)
    print(f'Q: {sample} -> Intent: {tag} ({conf*100:.1f}%) | Response: {resp}')